# 2. EDA e Feature Space

Questo notebook controlla correlazioni, distribuzioni principali e costruisce gli spazi usati dal clustering:

- **Set A**: feature planetarie/orbitali;
- **Set B**: Set A + feature della stella ospite.

Le label (`planet_type`, `star_type`) sono usate solo per colorare le proiezioni PCA, non per addestrare i cluster.


In [1]:
from pathlib import Path
import os

# Make the notebook robust both when executed from the project root and from
# notebook_final.
if Path.cwd().name != "notebook_final" and (Path.cwd() / "notebook_final").exists():
    os.chdir(Path.cwd() / "notebook_final")

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_PATH = PROJECT_ROOT / "input" / "nasa_exoplanet_intelligence.csv"

import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

processed_dir = Path("data/processed")
df_master = pd.read_csv(processed_dir / "df_master.csv")
X_imputed = pd.read_csv(processed_dir / "X_imputed.csv")
X_scaled = pd.read_csv(processed_dir / "X_scaled.csv")

with open(processed_dir / "preprocessing_metadata.json", encoding="utf-8") as f:
    prep_meta = json.load(f)

print(f"X_scaled shape: {X_scaled.shape}")
print("Feature disponibili:")
print(list(X_scaled.columns))


X_scaled shape: (6150, 16)
Feature disponibili:
['equilibrium_temp_k', 'orbital_eccentricity', 'orbital_period_days_log', 'planet_radius_earth_log', 'planet_mass_earth_log', 'semi_major_axis_au_log', 'star_temp_k', 'star_radius_sun', 'star_mass_sun', 'star_age_gyr', 'star_surface_gravity', 'star_metallicity', 'n_stars', 'n_planets', 'multi_planet_system', 'dist_from_earth_pc_log']


## 2.1 Correlazioni


In [2]:
corr = X_imputed.corr(numeric_only=True)

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, vmin=-1, vmax=1, linewidths=0.4)
plt.title("Matrice di correlazione delle feature numeriche")
plt.tight_layout()
plt.show()

corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().reset_index()
corr_pairs.columns = ["feature_1", "feature_2", "correlation"]
corr_pairs["abs_corr"] = corr_pairs["correlation"].abs()
print("Top 15 coppie piu' correlate:")
display(corr_pairs.sort_values("abs_corr", ascending=False).head(15).round(3))


Top 15 coppie piu' correlate:


,feature_1,feature_2,correlation,abs_corr
52,planet_radius_earth_log,planet_mass_earth_log,0.892,0.892
222,n_planets,multi_planet_system,0.776,0.776
122,star_radius_sun,star_surface_gravity,-0.720,0.720
138,star_mass_sun,star_surface_gravity,-0.602,0.602
69,planet_mass_earth_log,semi_major_axis_au_log,0.516,0.516
37,orbital_period_days_log,semi_major_axis_au_log,0.508,0.508
36,orbital_period_days_log,planet_mass_earth_log,0.461,0.461
120,star_radius_sun,star_mass_sun,0.459,0.459
18,orbital_eccentricity,orbital_period_days_log,0.451,0.451
20,orbital_eccentricity,planet_mass_earth_log,0.425,0.425


## 2.2 Distribuzioni delle feature chiave


In [3]:
key_features = [
    "orbital_period_days_log",
    "planet_radius_earth_log",
    "planet_mass_earth_log",
    "semi_major_axis_au_log",
    "equilibrium_temp_k",
    "orbital_eccentricity",
    "star_temp_k",
    "star_mass_sun",
]
key_features = [c for c in key_features if c in X_imputed.columns]

n_cols = 4
n_rows = int(np.ceil(len(key_features) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()
for ax, col in zip(axes, key_features):
    ax.hist(X_imputed[col].dropna(), bins=45, color="steelblue", edgecolor="white")
    ax.set_title(col)
for ax in axes[len(key_features):]:
    ax.set_visible(False)
plt.suptitle("Distribuzioni feature chiave (post log-transform, pre-scaling)", y=1.02)
plt.tight_layout()
plt.show()


## 2.3 Feature Set A/B


In [4]:
FEATURE_SET_A = [c for c in prep_meta["planetary_orbital_features"] if c in X_scaled.columns]
FEATURE_SET_B = FEATURE_SET_A + [
    c for c in prep_meta["stellar_features"]
    if c in X_scaled.columns and c not in FEATURE_SET_A
]

X_A = X_scaled[FEATURE_SET_A].copy()
X_B = X_scaled[FEATURE_SET_B].copy()

feature_dir = processed_dir / "feature_sets"
feature_dir.mkdir(parents=True, exist_ok=True)
X_A.to_csv(feature_dir / "X_A.csv", index=False)
X_B.to_csv(feature_dir / "X_B.csv", index=False)

feature_sets_metadata = {
    "feature_set_A_name": "A_planetary_orbital",
    "feature_set_A": FEATURE_SET_A,
    "feature_set_B_name": "B_planetary_orbital_stellar",
    "feature_set_B": FEATURE_SET_B,
    "excluded_from_clustering": [
        "planet_type",
        "star_type",
        "orbital_period_cat",
        "dist_category",
        "habitable_zone_flag",
        "ra",
        "dec",
        "disc_year",
        "discovery_method",
        "disc_facility",
    ],
}
with open(feature_dir / "feature_sets_metadata.json", "w", encoding="utf-8") as f:
    json.dump(feature_sets_metadata, f, indent=4, ensure_ascii=False)

print(f"Feature Set A ({X_A.shape[1]}):")
print(FEATURE_SET_A)
print(f"\nFeature Set B ({X_B.shape[1]}):")
print(FEATURE_SET_B)


Feature Set A (6):
['equilibrium_temp_k', 'orbital_eccentricity', 'orbital_period_days_log', 'planet_radius_earth_log', 'planet_mass_earth_log', 'semi_major_axis_au_log']

Feature Set B (12):
['equilibrium_temp_k', 'orbital_eccentricity', 'orbital_period_days_log', 'planet_radius_earth_log', 'planet_mass_earth_log', 'semi_major_axis_au_log', 'star_temp_k', 'star_radius_sun', 'star_mass_sun', 'star_age_gyr', 'star_surface_gravity', 'star_metallicity']


## 2.4 PCA come supporto visuale


In [5]:
def pca_frame(X, label):
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X)
    out = pd.DataFrame({"PC1": coords[:, 0], "PC2": coords[:, 1]})
    out["feature_set"] = label
    return out, pca.explained_variance_ratio_.sum()

sets = [(X_A, "Set A"), (X_B, "Set B")]
for hue_col, palette in [("planet_type", "tab10"), ("star_type", "Set2")]:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    for ax, (X, name) in zip(axes, sets):
        coords, var_exp = pca_frame(X, name)
        coords[hue_col] = df_master[hue_col].values
        sns.scatterplot(
            data=coords,
            x="PC1",
            y="PC2",
            hue=hue_col,
            palette=palette,
            alpha=0.55,
            s=18,
            ax=ax,
            legend=(ax is axes[-1]),
        )
        ax.set_title(f"{name} - PCA 2D, varianza={var_exp:.1%}")
    axes[-1].legend(title=hue_col, bbox_to_anchor=(1.03, 1), loc="upper left", fontsize=8)
    plt.suptitle(f"PCA colorata per {hue_col} (solo supporto interpretativo)")
    plt.tight_layout()
    plt.show()
